# UCI HAR Preprocessing

Prepare the UCI HAR example dataset for ZARA. This notebook loads raw inertial signals, builds a balanced test subset, saves serialized dataset files, and computes retrieval embeddings.


In [ ]:
import os
import pickle
import pandas as pd
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


## Activity Labels

Define the activity-name mapping used consistently by preprocessing, feature importance, and inference notebooks.


In [ ]:
activity_map = {
    1: 'Walking',
    2: 'Walking upstairs',
    3: 'Walking downstairs',
    4: 'Sitting',
    5: 'Standing',
    6: 'Laying'
}
label2id = {}
id2label = {}
for i, (key, value) in enumerate(activity_map.items()):
    label2id[value]=i
    id2label[i]=value
print(f"label2id:\n{label2id}")
print(f"id2label:\n{id2label}")


## Raw Dataset Loader

Load UCI train/test splits and stack accelerometer and gyroscope channels into fixed-shape sensor windows.


In [ ]:
def load_uci(datapath, split):
    """Load one UCI HAR split and stack the selected inertial signal channels."""
    split_path = os.path.join(datapath, split)
    label_path = os.path.join(split_path, "y_" + split + ".txt")
    label = np.loadtxt(label_path)
    subject_path = os.path.join(split_path, "subject_" + split + ".txt")
    subject = np.loadtxt(subject_path)

    assert len(subject) == len(label)

    signal_path = os.path.join(split_path, "Inertial Signals")
    channel_files_filtered = [f'total_acc_x_{split}.txt', f'total_acc_y_{split}.txt', f'total_acc_z_{split}.txt', f'body_gyro_x_{split}.txt', f'body_gyro_y_{split}.txt', f'body_gyro_z_{split}.txt']
    print(channel_files_filtered)

    datalist = []
    for c in channel_files_filtered:
        channel_data = np.loadtxt(os.path.join(signal_path, c))
        datalist.append(channel_data)
    data = np.stack(datalist, axis=2)
    return data, label, subject


## Build Segment Lists

Convert raw UCI windows into database and test segments with subject and activity metadata.


In [ ]:
all_database_segments = []
all_test_segments = []
all_database_labels = []
all_test_labels = []

for split in ["train", "test"]:
    datapath = './UCIDataset'
    data_lists, label_lists, subject_lists = load_uci(datapath, split)
    print(data_lists.shape)

    for i, (d, label, user_id) in enumerate(zip(data_lists, label_lists, subject_lists)):
        activity = label
        assert activity in activity_map.keys()
        subject = user_id

        assert len(d) ==128, f"{len(d)}"
        assert len(d[0]) == 6, f"{len(d[0])}"

        # Check for missing values.
        has_null = any(any(pd.isnull(item) or item == '' for item in sublist) for sublist in d)

        # Report missing values before failing.
        if has_null:
            i=0
            for sublist in d:
                 for item in sublist:
                     if pd.isnull(item) or item == '':
                         print(sublist)
                         i+=1
            print(i)
            raise ValueError("Input sequence contains missing values.")

        if split == 'train':
            all_database_segments.append(d)
        else:
            all_test_segments.append(d)

        label_dict = {
            "subject": int(subject),
            "activity_name": activity_map[activity],
            "activity": label2id[activity_map[activity]]
        }
        if split == 'train':
            all_database_labels.append(label_dict)
        else:
            all_test_labels.append(label_dict)

print(f"all_train_segments: {len(all_database_segments)}")
print(f"all_train_labels: {len(all_database_labels)}")
print(f"all_test_segments: {len(all_test_segments)}")
print(f"all_test_labels: {len(all_test_labels)}")


## Test Label Distribution

Inspect the raw test split activity distribution before balanced sampling.


In [ ]:
from collections import Counter

# Extract activity labels.
test_activity_labels = [label['activity_name'] for label in all_test_labels]

# Count samples per activity.
activity_distribution = Counter(test_activity_labels)

# Print sample counts per activity.
for activity_id, count in activity_distribution.items():
    print(f"{activity_id}: {count}")


## Balanced Test Subset

Select a subject-aware balanced subset of test examples for reproducible inference evaluation.


In [ ]:
import random
from collections import defaultdict

def split_balanced_by_activity_subject(all_labels, N, seed=42):
    """
    all_labels: list of dicts, each with keys "activity" and "subject"
    N: total number of samples to select per activity
    Returns: list of selected indices (length ≈ N * #activities)
    """
    random.seed(seed)

    # Group indices by activity → subject → [indices]
    by_act = defaultdict(lambda: defaultdict(list))
    for i, lbl in enumerate(all_labels):
        by_act[lbl["activity"]][lbl["subject"]].append(i)

    selected_indices = []

    for activity, subj2inds in by_act.items():
        # Flatten into (subject, [indices]) list
        subject_pools = [(subj, inds[:]) for subj, inds in subj2inds.items()]
        random.shuffle(subject_pools)

        # Shuffle within each subject
        for _, inds in subject_pools:
            random.shuffle(inds)

        total_collected = 0
        temp_selected = []

        # 1st pass: try to fairly distribute quota
        S = len(subject_pools)
        base, rem = divmod(N, S)

        # Try base + (1 if rem > 0) per subject
        for i, (subj, inds) in enumerate(subject_pools):
            quota = base + (1 if i < rem else 0)
            taken = inds[:quota]
            temp_selected.extend(taken)
            total_collected += len(taken)
            # update leftover
            subject_pools[i] = (subj, inds[quota:])

        # 2nd pass: top up remaining from any subject with leftovers
        if total_collected < N:
            remaining = N - total_collected
            flat_leftover = [idx for _, inds in subject_pools for idx in inds]
            random.shuffle(flat_leftover)
            temp_selected.extend(flat_leftover[:remaining])

        selected_indices.extend(temp_selected)

    return selected_indices

# usage:
set_idx = split_balanced_by_activity_subject(
    all_test_labels,
    N=40,    # e.g. 25 per activity in split
    seed=SEED
)

all_test_segments_final = [all_test_segments[i] for i in set_idx]
all_test_labels_final   = [all_test_labels[i]   for i in set_idx]

from collections import defaultdict, Counter

# Count activity samples by subject.
activity_subject_counts1 = defaultdict(Counter)
for lbl in all_test_labels_final:
    act = lbl["activity"]
    subj = lbl["subject"]
    activity_subject_counts1[act][subj] += 1

# Print the subject distribution for each activity.
for act, subj_counts in activity_subject_counts1.items():
    print(f"Activity = {act}")
    for subj, cnt in subj_counts.items():
        print(f"    Subject {subj}: {cnt}")
    print()


## Save Processed Dataset

Persist processed database and test examples for downstream notebooks.


In [ ]:
output_path = "./dataset/uci/"

with open(os.path.join(output_path, 'uci_database_segments.pkl'), 'wb') as f:
    pickle.dump(all_database_segments, f)

with open(os.path.join(output_path, 'uci_test_data.pkl'), 'wb') as f:
    pickle.dump(all_test_segments_final, f)

with open(os.path.join(output_path, 'uci_database_labels.pkl'), 'wb') as f:
    pickle.dump(all_database_labels, f)

with open(os.path.join(output_path, 'uci_test_labels.pkl'), 'wb') as f:
    pickle.dump(all_test_labels_final, f)


## Load Time-Series Encoder

Load the Mantis encoder used to compute retrieval embeddings.


In [ ]:
from mantis.trainer import MantisTrainer

from mantis.architecture import Mantis8M

network = Mantis8M(device='mps')
network = network.from_pretrained("paris-noah/Mantis-8M")

model = MantisTrainer(device='mps', network=network)


## Embedding Helper

Define the batched embedding function for multi-channel UCI windows.


In [ ]:
import torch
import torch.nn.functional as F

def get_multi_channel_embeddings(data_array):
    """Compute frozen time-series encoder embeddings for multi-channel windows."""
    data_array = np.array(data_array, dtype=np.float32)
    data_array = data_array.transpose(0, 2, 1)

    num_samples, num_channels, seq_len = data_array.shape
    print(f"num_samples: {num_samples}; Channel num: {num_channels}; seq_len: {seq_len}")

    data_torch = torch.tensor(data_array, dtype=torch.float32, device="mps")
    data_torch_scaled = F.interpolate(data_torch, size=512, mode='linear', align_corners=False)

    embeddings = model.transform(data_torch_scaled)

    # Return embeddings as a NumPy array.
    return embeddings


## Compute Database Embeddings

Embed all database windows with the frozen encoder.


In [ ]:
all_database_embeddings = get_multi_channel_embeddings(all_database_segments)
print("all_database_embeddings shape:", all_database_embeddings.shape)


## Save Embeddings

Save database embeddings for the retrieval-based inference notebook.


In [ ]:
np.save(os.path.join(output_path, "all_database_embeddings.npy"), all_database_embeddings)
